# 09 — Build glacier masks from GLIMS outlines

**Purpose.** Makes the training labels: one binary mask per glacier, burned from its GLIMS outline
(`glacierview.training.rasterize_outline`). Scenes of the same glacier can fall in different UTM
zones, so each outline is reprojected into the CRS of the glacier's first scene listed in
`image_attributes.csv`, and masked on that scene's grid (1 inside the outline, 0 outside). Notebook 10
resizes it to 128×128.

## Inputs
- **`config.glims_selection_path()`**: GLIMS outlines from notebook 01.
- **`image_attributes.csv`** in `config.processed_metadata_dir(config.TRAINING_DATA_LABEL)`: from
  notebook 06; picks the scene each mask is drawn on.
- **The training scenes**: `config.landsat_dir(config.TRAINING_DATA_LABEL)`, from notebook 03.

## Outputs
- `config.masks_dir()/<glims_id>.tif` (`src/segmentation/training/data/masks_staging_2/`): one mask
  per glacier, at the scene's full resolution. Gitignored, with no S3 copy.

In [ ]:
import pandas as pd
from PIL import Image
from tqdm import tqdm

from glacierview import config, glims
from glacierview.training import rasterize_outline

LABEL = config.TRAINING_DATA_LABEL
attributes = pd.read_csv(config.processed_metadata_dir(LABEL) / "image_attributes.csv",
                         usecols=["file_name"])
first_scene = (attributes.assign(glims_id=attributes["file_name"].str.split("_").str[0])
               .drop_duplicates("glims_id").set_index("glims_id")["file_name"])
outlines = glims.read_selection().set_index("glac_id").geometry

In [ ]:
out = config.masks_dir()
out.mkdir(parents=True, exist_ok=True)
for glims_id in tqdm(first_scene.index.intersection(outlines.index)):
    scene = config.glacier_dir(glims_id, LABEL) / first_scene[glims_id]
    Image.fromarray(rasterize_outline(scene, outlines[glims_id])).save(out / f"{glims_id}.tif")